# Paper author countries — `countries`, `n_countries`, first- and last-author country

The OpenAlex twin of the `countries` / `n_countries` columns in
`Dimensions/output/paper_author.parquet`, and the sibling of `paper_author.ipynb`, whose read
path it copies exactly (that notebook finished in 38 min on a 250 GB `jevans` node, so the
DuckDB scan of this file is a proven route — it is **not** a login-node job).

## Source
```
/project/jevans/renli_shared/OpenAlex_2026_Jan_16_Renly_parquet/works_au_affs_fixed.csv.gz
```
40.8 GB gzip, one row per (work, author, affiliation). Four columns are used: `work_id`,
`author_id`, `author_position_int`, `countries`.

## What `countries` is, and what it is not
`countries` is OpenAlex's **author-level** list of ISO2 codes for that authorship — `"CN; US"`
for an author with a Chinese and a US affiliation — and it is repeated on every affiliation row
of that author. Measured on a 300,000-row sample, 18 of 246,493 (work, author) pairs carried the
list on one row and a blank on another; the union over an author's rows below absorbs that.
About a third of rows have no country at all (no affiliation was matched to an institution).

Consequences for the columns:
- an author counts **once** towards `team_size` and `n_located` whatever the number of rows or
  countries — the de-duplication on `(work_id, author_id)` from `paper_author.ipynb` is applied
  first, so `team_size` here must equal `team_size` there (asserted below);
- an author with two countries contributes to **both** entries of `country_author_counts`, so
  that column can sum to more than `n_located`;
- Namibia is `NA`. DuckDB's CSV reader treats only the empty string as NULL, so it survives;
  pandas would not have kept it.

## Output — `OpenAlex/output/paper_author_country.parquet`
| column | |
|---|---|
| `paper_id` | OpenAlex work id, `W…`, prefix stripped — keyed like the other outputs here (`paper_author.parquet` calls the same column `work_id`) |
| `team_size` | distinct authors (identical to `paper_author.team_size`) |
| `n_located` | authors with at least one country |
| `countries` | distinct ISO2 codes over all authors, **sorted**, `;`-joined; null if no author is located |
| `n_countries` | `len(countries.split(';'))`, 0 when null |
| `country_author_counts` | located authors per country, `US:3;CN:1`, ordered by count desc then code |
| `first_author_country` / `last_author_country` | the country list of the lowest / highest-position author, `;`-joined; null if *that* author is unlocated (not promoted to the next located one) |
| `is_international` | `n_countries > 1` |

Two passes: (1) the CSV scan writes one row per (work, author) with its country list to
`cache/author_country_by_author.parquet`; (2) the per-paper table is aggregated from that
parquet, which is what lets the per-country counts be a second grouping instead of a second
40 GB scan. Environment hooks for a smoke test: `NB_AU_AFFS_SRC` (a small CSV with the same
header), `NB_OUT_DIR`, `NB_DUCKDB_MEM`.

In [ ]:
import os, sys, time, gc
import numpy as np, pandas as pd
import duckdb
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/OpenAlex')
import oa_common as oa

SRC      = os.environ.get('NB_AU_AFFS_SRC', f'{oa.ROOT}/works_au_affs_fixed.csv.gz')
OUT_DIR  = os.environ.get('NB_OUT_DIR', oa.OUT)
SMOKE    = 'NB_AU_AFFS_SRC' in os.environ
OUT_FP   = f'{OUT_DIR}/paper_author_country.parquet'
AUTH_FP  = f'{OUT_DIR if SMOKE else oa.CACHE}/author_country_by_author.parquet'   # pass-1 artefact
PA_FP    = f'{oa.OUT}/paper_author.parquet'      # sibling, cross-check only
META_FP  = f'{oa.OUT}/paper_metadata.parquet'    # sibling, year for the coverage table only
os.makedirs(OUT_DIR, exist_ok=True)

# Same explicit VARCHAR schema as paper_author.ipynb: a 40.8 GB gzip is sniffed from its first
# rows only, and affiliation_json carries embedded commas and quotes.
COLS = {c: 'VARCHAR' for c in [
    'work_id', 'author_position', 'author_position_raw', 'author_position_int', 'author_id',
    'author_display_name', 'raw_author_name', 'is_corresponding', 'affiliation_id',
    'raw_affiliation_string', 'institution_ids', 'countries', 'type', 'orcid',
    'affiliation_json']}

MEM = os.environ.get('NB_DUCKDB_MEM', '180GB')
con = duckdb.connect()
con.execute(f"SET memory_limit='{MEM}'")
con.execute(f"SET temp_directory='{oa.CACHE}/duckdb_tmp_author_country'")
con.execute("SET preserve_insertion_order=false")
con.execute("SET enable_progress_bar=false")
print(f'source : {SRC}  ({os.path.getsize(SRC)/1e9:.2f} GB)' + ('   *** SMOKE TEST' if SMOKE else ''))
print(f'pass 1 : {AUTH_FP}')
print(f'output : {OUT_FP}')
print(f'duckdb : memory_limit {MEM}')

## 1. Pass 1 — one row per (work, author) with its country list

`countries` is split on `;` (spaces stripped), flattened over the author's rows, de-duplicated
and sorted. `clist` is an empty list, not NULL, for an unlocated author, so `len(clist) > 0`
is the "located" test everywhere below.

In [ ]:
%%time
t0 = time.time()
if os.path.exists(AUTH_FP) and not SMOKE and not os.environ.get('NB_FORCE_PASS1'):
    print(f'pass 1 already written ({os.path.getsize(AUTH_FP)/1e9:.2f} GB) -- skipping; set NB_FORCE_PASS1=1 to rebuild')
else:
  con.execute(f"""
COPY (
  WITH raw AS (
    SELECT replace(work_id,   'https://openalex.org/', '') AS paper_id,
           replace(author_id, 'https://openalex.org/', '') AS author_id,
           TRY_CAST(author_position_int AS INTEGER)        AS pos,
           nullif(replace(countries, ' ', ''), '')         AS countries
    FROM read_csv('{SRC}', header=true, columns={COLS})
    WHERE author_id IS NOT NULL AND author_id <> '')
  SELECT paper_id, author_id, min(pos) AS pos,
         coalesce(list_sort(list_distinct(flatten(list(str_split(countries, ';')) FILTER (WHERE countries IS NOT NULL)))), []) AS clist
  FROM raw
  GROUP BY paper_id, author_id
) TO '{AUTH_FP}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 2000000)""")
  os.replace(AUTH_FP + '.tmp', AUTH_FP)
n_auth = con.execute(f"SELECT count(*) FROM read_parquet('{AUTH_FP}')").fetchone()[0]
print(f'WROTE {AUTH_FP}  ({os.path.getsize(AUTH_FP)/1e9:.2f} GB, {n_auth:,} (work, author) rows) in {time.time()-t0:.0f}s')

## 2. Pass 2 — the per-paper table, in four spillable steps

The obvious single query — `list(clist ORDER BY pos)` per paper over 738M rows — was killed at
168 GiB on a 250 GB node: DuckDB cannot offload `list()` / `string_agg()` aggregate states to
disk, so 300M+ groups of nested lists had to fit in RAM at once. Pass 2 is therefore split so that
every heavy grouping uses only aggregates that spill (`count`, `min`, `max`):

1. **scalars** per paper: `team_size`, `n_located`, and the order keys of the first and last
   author (`okey = pos·10¹⁰ + numeric author id`, unique within a paper);
2. **first / last author country** by joining those two keys back to the author table and taking
   `max(CASE WHEN okey = kmin …)` — no list, no ORDER BY inside an aggregate;
3. **authors per (paper, country)** via `unnest(clist)` + `count(*)`;
4. **assemble**: only this step, which turns the (paper, country) rows into `countries` /
   `country_author_counts`, uses `string_agg`, over ~250M small groups.

Pass 1 is skipped when its parquet already exists (`NB_FORCE_PASS1=1` rebuilds it).

In [ ]:
%%time
t0 = time.time()
TMP_DIR = OUT_DIR if SMOKE else oa.CACHE
P0_FP, P_FP, PC_FP = (f'{TMP_DIR}/author_country_{x}.tmp.parquet' for x in ('p0', 'p', 'pc'))
A = (f"(SELECT paper_id, author_id, clist, "
     f"coalesce(pos, 2147483647)::BIGINT * 10000000000 + coalesce(TRY_CAST(substr(author_id, 2) AS BIGINT), 0) AS okey "
     f"FROM read_parquet('{AUTH_FP}'))")

# 1. spillable scalars per paper
con.execute(f"""
COPY (SELECT paper_id, count(*) AS team_size, count(*) FILTER (WHERE len(clist) > 0) AS n_located,
             min(okey) AS kmin, max(okey) AS kmax
      FROM {A} a GROUP BY paper_id) TO '{P0_FP}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
print(f'  [{time.time()-t0:.0f}s] scalars per paper written')

# 2. first / last author's country list, by key lookup
con.execute(f"""
COPY (SELECT p.paper_id, any_value(p.team_size) AS team_size, any_value(p.n_located) AS n_located,
             max(CASE WHEN a.okey = p.kmin THEN array_to_string(a.clist, ';') END) AS first_c,
             max(CASE WHEN a.okey = p.kmax THEN array_to_string(a.clist, ';') END) AS last_c
      FROM read_parquet('{P0_FP}') p JOIN {A} a ON a.paper_id = p.paper_id AND (a.okey = p.kmin OR a.okey = p.kmax)
      GROUP BY p.paper_id) TO '{P_FP}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
print(f'  [{time.time()-t0:.0f}s] first / last author country written')

# 3. authors per (paper, country)
con.execute(f"""
COPY (SELECT paper_id, c AS country, count(*) AS n
      FROM read_parquet('{AUTH_FP}'), unnest(clist) AS u(c) GROUP BY 1, 2) TO '{PC_FP}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
print(f'  [{time.time()-t0:.0f}s] (paper, country) counts written')

# 4. assemble
con.execute(f"""
COPY (
  WITH cagg AS (
    SELECT paper_id, string_agg(country, ';' ORDER BY country) AS countries, count(*) AS n_countries,
           string_agg(country || ':' || n, ';' ORDER BY n DESC, country) AS country_author_counts
    FROM read_parquet('{PC_FP}') GROUP BY paper_id)
  SELECT p.paper_id,
         p.team_size::INTEGER                   AS team_size,
         p.n_located::INTEGER                   AS n_located,
         c.countries,
         coalesce(c.n_countries, 0)::INTEGER    AS n_countries,
         c.country_author_counts,
         nullif(p.first_c, '')                  AS first_author_country,
         nullif(p.last_c, '')                   AS last_author_country,
         coalesce(c.n_countries, 0) > 1         AS is_international
  FROM read_parquet('{P_FP}') p LEFT JOIN cagg c USING (paper_id)
  ORDER BY p.paper_id
) TO '{OUT_FP}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)""")
os.replace(OUT_FP + '.tmp', OUT_FP)
for f in (P0_FP, P_FP, PC_FP):
    os.remove(f)
n_papers = con.execute(f"SELECT count(*) FROM read_parquet('{OUT_FP}')").fetchone()[0]
print(f'WROTE {OUT_FP}  ({os.path.getsize(OUT_FP)/1e9:.2f} GB, {n_papers:,} papers) in {time.time()-t0:.0f}s')

## 3. Verification

1. `paper_id` unique; `n_located ≤ team_size`; `n_countries == len(countries.split(';'))`;
   the list is sorted and duplicate-free.
2. `country_author_counts` names exactly the codes in `countries`, and its sum is ≥ `n_located`
   (equal unless some author has two countries — that surplus is reported, not asserted).
3. `first_author_country` / `last_author_country` ⊆ `countries`; `is_international` agrees with
   `n_countries`.
4. Against `paper_author.parquet`: same set of works and the same `team_size` on every one of
   them. Both files de-duplicate on `(work_id, author_id)` from the same source, so any
   difference is a defect. (Skipped in a smoke test: it scans 251M rows.)
5. Coverage: located share overall, by team size and by publication year (via
   `paper_metadata.year`), and the most frequent countries.

In [ ]:
%%time
chk = con.execute(f"""
WITH s AS (SELECT * FROM read_parquet('{OUT_FP}')),
     x AS (SELECT *, CASE WHEN countries IS NULL THEN [] ELSE str_split(countries, ';') END AS parts,
                     CASE WHEN country_author_counts IS NULL THEN []
                          ELSE list_transform(str_split(country_author_counts, ';'), z -> str_split(z, ':')) END AS cc,
                     CASE WHEN first_author_country IS NULL THEN [] ELSE str_split(first_author_country, ';') END AS fparts,
                     CASE WHEN last_author_country  IS NULL THEN [] ELSE str_split(last_author_country,  ';') END AS lparts
           FROM s)
SELECT count(*)                                                                  AS papers,
       count(*) - count(DISTINCT paper_id)                                       AS dup_paper_id,
       sum(team_size)                                                            AS author_slots,
       sum(n_located)                                                            AS located_slots,
       count(*) FILTER (WHERE n_located > team_size)                             AS located_gt_team,
       count(*) FILTER (WHERE n_countries <> len(parts))                         AS n_countries_mismatch,
       count(*) FILTER (WHERE len(parts) <> len(list_distinct(parts)))           AS dup_in_countries,
       count(*) FILTER (WHERE parts <> list_sort(parts))                         AS unsorted_countries,
       count(*) FILTER (WHERE list_sort(list_transform(cc, z -> z[1])) <> parts)  AS counts_codes_mismatch,
       count(*) FILTER (WHERE coalesce(list_sum(list_transform(cc, z -> z[2]::INTEGER)), 0) < n_located) AS counts_sum_lt_located,
       count(*) FILTER (WHERE coalesce(list_sum(list_transform(cc, z -> z[2]::INTEGER)), 0) > n_located) AS multi_country_authors_papers,
       count(*) FILTER (WHERE NOT list_has_all(parts, fparts))                   AS first_not_in_list,
       count(*) FILTER (WHERE NOT list_has_all(parts, lparts))                   AS last_not_in_list,
       count(*) FILTER (WHERE is_international <> (n_countries > 1))             AS intl_mismatch,
       count(*) FILTER (WHERE countries IS NOT NULL)                             AS with_country,
       count(*) FILTER (WHERE is_international)                                  AS international
FROM x""").fetchdf().iloc[0]
for k, v in chk.items():
    print(f'  {k:<30} {int(v):>15,}')
ok = all(chk[k] == 0 for k in ['dup_paper_id', 'located_gt_team', 'n_countries_mismatch', 'dup_in_countries',
                                'unsorted_countries', 'counts_codes_mismatch', 'counts_sum_lt_located',
                                'first_not_in_list', 'last_not_in_list', 'intl_mismatch'])
print(f"\n1-3. {'ALL PASS' if ok else 'FAILED -- see the counts above'}")
print(f'     located share: {chk.with_country/chk.papers*100:.1f}% of papers, '
      f'{chk.located_slots/chk.author_slots*100:.1f}% of author slots; international: {chk.international/chk.papers*100:.1f}%; '
      f'papers with a multi-country author: {int(chk.multi_country_authors_papers):,}')

if SMOKE:
    print('4.   smoke test: the paper_author.parquet cross-check (251M-row scan) is skipped')
elif os.path.exists(PA_FP):
    m = con.execute(f"""
    WITH n AS (SELECT paper_id, team_size FROM read_parquet('{OUT_FP}')),
         o AS (SELECT work_id AS paper_id, team_size FROM read_parquet('{PA_FP}'))
    SELECT (SELECT count(*) FROM n JOIN o USING (paper_id))                                 AS shared,
           (SELECT count(*) FROM n JOIN o USING (paper_id) WHERE n.team_size <> o.team_size) AS team_size_mismatch,
           (SELECT count(*) FROM n ANTI JOIN o USING (paper_id))                            AS new_only,
           (SELECT count(*) FROM o ANTI JOIN n USING (paper_id))                            AS old_only""").fetchdf().iloc[0]
    for k, v in m.items():
        print(f'  {k:<30} {int(v):>15,}')
    ok4 = m.team_size_mismatch == 0 and m.new_only == 0 and m.old_only == 0
    print(f"4.   vs paper_author.parquet: {'OK' if ok4 else 'FAILED'}")
else:
    print('4.   paper_author.parquet not present -- cross-check skipped')

print('\n5.   coverage by team size')
display(con.execute(f"""SELECT team_size, count(*) AS papers,
                        round(100.0*count(countries)/count(*), 1) AS pct_located,
                        round(100.0*count(*) FILTER (WHERE is_international)/count(*), 2) AS pct_international
                        FROM read_parquet('{OUT_FP}') GROUP BY 1 ORDER BY 1 LIMIT 12""").fetchdf())
if os.path.exists(META_FP) and not SMOKE:
    print('     coverage by publication decade (paper_metadata.year)')
    display(con.execute(f"""
    SELECT (m.year::INTEGER // 10) * 10 AS decade, count(*) AS papers,
           round(100.0*count(s.countries)/count(*), 1) AS pct_located,
           round(100.0*count(*) FILTER (WHERE s.is_international)/count(*), 2) AS pct_international
    FROM read_parquet('{OUT_FP}') s JOIN read_parquet('{META_FP}') m USING (paper_id)
    WHERE m.year IS NOT NULL AND m.year BETWEEN 1900 AND 2026 GROUP BY 1 ORDER BY 1""").fetchdf())
print('     most frequent countries (papers with >=1 author there)')
display(con.execute(f"""
SELECT c AS country, count(*) AS papers, round(100.0*count(*)/(SELECT count(*) FROM read_parquet('{OUT_FP}')), 2) AS pct
FROM read_parquet('{OUT_FP}'), unnest(str_split(countries, ';')) AS u(c)
WHERE countries IS NOT NULL GROUP BY 1 ORDER BY 2 DESC LIMIT 15""").fetchdf())
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') WHERE is_international ORDER BY n_countries DESC LIMIT 5").fetchdf())
con.close()